In [1]:
import os
import sqlite3
import pandas as pd

print(sqlite3.sqlite_version)

3.50.4


In [2]:
ROOT = r'D:\Bluestock work\bluestock-fintech-analysis1'

SQL_FILE = os.path.join(ROOT, 'sql', '01_star_schema.sql')
DB_PATH = os.path.join(ROOT, 'data', 'processed', 'bluestock_dw.db')

with open(SQL_FILE, 'r', encoding='utf-8') as f:
    schema_sql = f.read()

conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON')
conn.executescript(schema_sql)
conn.commit()

print('Database made:', DB_PATH)

Database made: D:\Bluestock work\bluestock-fintech-analysis1\data\processed\bluestock_dw.db


In [3]:
pd.read_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'index') ORDER BY type, name",
    conn,
)

,name,type
0,idx_dim_fund_category,index
1,idx_dim_fund_house,index
2,idx_fact_aum_house,index
3,idx_fact_nav_date,index
4,idx_fact_tx_date,index
5,idx_fact_tx_fund,index
6,idx_fact_tx_investor,index
7,sqlite_autoindex_dim_date_1,index
8,sqlite_autoindex_fact_aum_1,index
9,sqlite_autoindex_fact_nav_1,index


In [4]:
pd.read_sql_query('PRAGMA table_info(fact_nav)', conn)

,cid,name,type,notnull,dflt_value,pk
0,0,amfi_code,INTEGER,1,NaN,1
1,1,date_key,INTEGER,1,NaN,2
2,2,nav,REAL,1,NaN,0
3,3,is_filled,INTEGER,1,0,0


In [6]:
for table in ['fact_nav', 'fact_transactions', 'fact_performance', 'fact_aum']:
    print(table)
    fk = pd.read_sql_query(f'PRAGMA foreign_key_list({table})', conn)
    print(fk[['from', 'table', 'to']])
    print()

fact_nav
        from     table         to
0   date_key  dim_date   date_key
1  amfi_code  dim_fund  amfi_code

fact_transactions
        from     table         to
0  amfi_code  dim_fund  amfi_code
1   date_key  dim_date   date_key

fact_performance
                from     table         to
0  snapshot_date_key  dim_date   date_key
1          amfi_code  dim_fund  amfi_code

fact_aum
       from     table        to
0  date_key  dim_date  date_key



In [7]:
test = sqlite3.connect(':memory:')
test.execute('PRAGMA foreign_keys = ON')
test.executescript(schema_sql)

test.execute("INSERT INTO dim_fund (amfi_code, fund_house, scheme_name) VALUES (100001, 'Test MF', 'Test Fund')")
test.execute("""INSERT INTO dim_date VALUES
    (20261009, '2026-10-09', 2026, 4, 10, 'October', 9, 4, 'Friday', 41, 0, 2027, 3)""")
test.commit()

def try_sql(label, sql, params=()):
    try:
        test.execute(sql, params)
        print('OK     :', label)
    except sqlite3.IntegrityError as error:
        print('BLOCKED:', label, '->', error)

In [8]:
nav_sql = 'INSERT INTO fact_nav (amfi_code, date_key, nav) VALUES (?, ?, ?)'
tx_sql = '''INSERT INTO fact_transactions
            (investor_id, date_key, amfi_code, transaction_type, amount_inr)
            VALUES (?, ?, ?, ?, ?)'''

try_sql('valid NAV row',                         nav_sql, (100001, 20261009, 100.5))
try_sql('same fund and date twice',              nav_sql, (100001, 20261009, 101.0))
try_sql('NAV below zero',                        nav_sql, (100001, 20261009, -5.0))
try_sql('fund not in dim_fund',                  nav_sql, (999999, 20261009, 50.0))
try_sql('date not in dim_date',                  nav_sql, (100001, 20261010, 50.0))
try_sql('transaction type Buy',                  tx_sql,  ('TEST1', 20261009, 100001, 'Buy', 500.0))
try_sql('amount zero',                           tx_sql,  ('TEST1', 20261009, 100001, 'SIP', 0))
try_sql('valid transaction',                     tx_sql,  ('TEST1', 20261009, 100001, 'SIP', 500.0))
try_sql('text in a number column',
        'INSERT INTO fact_performance (amfi_code, snapshot_date_key, return_1yr_pct) VALUES (?, ?, ?)',
        (100001, 20261009, 'abc'))
try_sql('delete a fund that has NAV rows',       'DELETE FROM dim_fund WHERE amfi_code = ?', (100001,))

OK     : valid NAV row
BLOCKED: same fund and date twice -> UNIQUE constraint failed: fact_nav.amfi_code, fact_nav.date_key
BLOCKED: NAV below zero -> CHECK constraint failed: nav > 0
BLOCKED: fund not in dim_fund -> FOREIGN KEY constraint failed
BLOCKED: date not in dim_date -> FOREIGN KEY constraint failed
BLOCKED: transaction type Buy -> CHECK constraint failed: transaction_type IN ('SIP', 'Lumpsum', 'Redemption')
BLOCKED: amount zero -> CHECK constraint failed: amount_inr > 0
OK     : valid transaction
BLOCKED: text in a number column -> cannot store TEXT value in REAL column fact_performance.return_1yr_pct
BLOCKED: delete a fund that has NAV rows -> FOREIGN KEY constraint failed


In [9]:
test.commit()
test.execute('PRAGMA foreign_keys = OFF')

try_sql('fund not in dim_fund, foreign keys OFF', nav_sql, (999999, 20261009, 50.0))

test.execute('PRAGMA foreign_keys = ON')

OK     : fund not in dim_fund, foreign keys OFF


In [10]:
test.close()
conn.close()

## Decisions
- dim_fund uses amfi_code as the primary key (natural key).
- dim_date uses date_key as an integer in the form YYYYMMDD.
- fact_transactions uses transaction_id as a surrogate key. The source file has no transaction ID.
- fact_performance has a snapshot_date_key because the source file has no date.
- fact_aum stores aum_crore only. aum_lakh_crore is the same value in another unit.